<a href="https://colab.research.google.com/github/sky-blue/manim/blob/main/manim_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!sudo apt update
!sudo apt install -y libcairo2-dev libpango1.0-dev ffmpeg texlive-latex-base texlive-fonts-recommended texlive-latex-extra
!pip install manim

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease
Hit:4 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:5 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:6 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:7 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu noble InRelease
Hit:10 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
109 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not see

In [5]:
%%writefile scene.py
from manim import *

class NormalizeBinary(Scene):
    def construct(self):
        self.camera.background_color = "#101526"
        print("STEP 1: Scene started")
        # Title
        title = Text(
            "Normalization",
            font_size=36,
            color=WHITE
        ).to_edge(UP).shift(DOWN * 0.25)

        self.play(Write(title))
        print("STEP 2: Title completed")
        self.wait(0.5)
        # 1. Initial binary representation
        dec_label = MathTex(
            r"0.1_{10} =",
            font_size=44
        )

        digits = [
            "0", ".", "0", "0", "0", "1",
            "1", "0", "0", "1", "1", "0", "0", r"\ldots"
        ]

        bin_mobs = VGroup(
            *[MathTex(d, font_size=44) for d in digits]
        ).arrange(RIGHT, buff=0.08)

        start_eq = VGroup(
            dec_label, bin_mobs
        ).arrange(RIGHT, buff=0.2)

        start_eq.move_to(UP * 0.3)

        # Lower the decimal point slightly
        bin_mobs[1].shift(DOWN * 0.13)

        self.play(Write(dec_label), Write(bin_mobs))
        print("STEP 3: Binary completed")
        self.wait(1)

        # 2. Highlight the first 1
        first_one = bin_mobs[5]

        self.play(
            first_one.animate.set_color(GREEN)
        )
        self.wait(0.5)

        # 3. Downward semicircular arrows

        # Downward semicircular arrows
        arrows = VGroup()

        arrow_y = 0.18

        for i in range(4):

            # Start point: midpoint between two digits
            start_x = (
                bin_mobs[i + 1].get_center()[0]
                + bin_mobs[i + 2].get_center()[0]
            ) / 2

            # End point: midpoint between the next two digits
            end_x = (
                bin_mobs[i + 2].get_center()[0]
                + bin_mobs[i + 3].get_center()[0]
            ) / 2

            arrow = CurvedArrow(
                start_point=[start_x, arrow_y, 0],
                end_point=[end_x, arrow_y, 0],
                angle=PI,
                color=YELLOW,
                stroke_width=3,
                tip_length=0.13
            )

            self.play(
                Create(arrow),
                run_time=0.45
            )

            arrows.add(arrow)


        # Shift count
        shift_text = MathTex(
            r"4\ shifts",
            font_size=30,
            color=YELLOW
        ).next_to(arrows, DOWN, buff=0.2)

        self.play(Write(shift_text))
        self.wait(1)



                # 4. Step-by-step normalization

        # Remove arrows, shift count, and original binary digits
        self.play(
            FadeOut(arrows),
            FadeOut(shift_text),
            FadeOut(bin_mobs),
            run_time=0.7
        )

        # Keep the decimal label and align expressions to its right
        fixed_left = dec_label.get_right()[0] + 0.2
        expression_y = 0.3

        step_strings = [
            "0.00011001100...",
            "0.0011001100...",
            "0.011001100...",
            "0.11001100...",
            "1.1001100..."
        ]

        current_expr = MathTex(
            step_strings[0],
            font_size=44
        )

        current_expr.move_to([
            fixed_left + current_expr.width / 2,
            expression_y,
            0
        ])

        self.play(Write(current_expr), run_time=0.8)
        self.wait(0.5)

        # Transform each step while keeping the left edge fixed
        for text in step_strings[1:]:
            next_expr = MathTex(
                text,
                font_size=44
            )

            next_expr.move_to([
                fixed_left + next_expr.width / 2,
                expression_y,
                0
            ])

            self.play(
                FadeTransform(current_expr, next_expr),
                run_time=0.8
            )

            current_expr = next_expr
            self.wait(0.5)
        # 5. Add exponent
        multiplier = MathTex(
            r"\times 2^{-4}",
            color=RED,
            font_size=44
        )

        multiplier.next_to(current_expr, RIGHT, buff=0.2)

        self.play(Write(multiplier))
        self.wait(1)


        # 6. Move the normalized expression upward

        normalized_equation = VGroup(current_expr, multiplier)

        self.play(
            normalized_equation.animate.move_to(UP * 1.65),
            run_time=1.2
        )

        # 7. Create the 64-bit storage area

        cell_width = 0.16
        cell_height = 0.42
        cell_gap = 0.012

        def make_slots(count):
            return VGroup(*[
                Rectangle(
                    width=cell_width,
                    height=cell_height,
                    stroke_color=WHITE,
                    stroke_width=1.2
                )
                for _ in range(count)
            ]).arrange(RIGHT, buff=cell_gap)

        sign_slots = make_slots(1)
        exponent_slots = make_slots(11)
        fraction_slots = make_slots(52)

        bit_fields = VGroup(
            sign_slots,
            exponent_slots,
            fraction_slots
        ).arrange(RIGHT, buff=0.18)

        bit_fields.move_to(DOWN * 1.35)

        sign_label = Text(
            "Sign",
            font_size=20,
            color=YELLOW
        ).next_to(sign_slots, DOWN, buff=0.15)

        exponent_label = Text(
            "Exponent",
            font_size=20,
            color=RED
        ).next_to(exponent_slots, DOWN, buff=0.15)

        fraction_label = Text(
            "Fraction",
            font_size=20,
            color=BLUE
        ).next_to(fraction_slots, DOWN, buff=0.15)

        self.play(
            Create(bit_fields),
            FadeIn(sign_label),
            FadeIn(exponent_label),
            FadeIn(fraction_label),
            run_time=1.5
        )

        self.wait(0.5)

        # 8. Remove the decimal expression and center the normalized number

        self.play(
            Unwrite(dec_label),
            normalized_equation.animate.move_to(UP * 1.65),
            run_time=1.2
        )

        self.wait(0.5)

        # 9. Fill the sign bit

        sign_bit = MathTex(
            "0",
            font_size=22,
            color=YELLOW
        ).move_to(sign_slots[0].get_center())

        self.play(Write(sign_bit))
        self.wait(0.5)

        # 10. Calculate the biased exponent

        bias_calc = MathTex(
            r"-4 + 1023 = 1019",
            font_size=30,
            color=RED
        ).move_to(UP * 0.45)

        self.play(Write(bias_calc))
        self.wait(1)

        exponent_bits = "01111111011"

        exponent_source = VGroup(*[
            MathTex(bit, font_size=20, color=RED)
            for bit in exponent_bits
        ]).arrange(RIGHT, buff=0.025)

        exponent_source.move_to(
            exponent_slots.get_center() + UP * 0.55
        )

        self.play(
            FadeOut(bias_calc),
            Write(exponent_source),
            run_time=0.8
        )

        self.play(
            LaggedStart(
                *[
                    bit.animate.move_to(slot.get_center())
                    for bit, slot in zip(exponent_source, exponent_slots)
                ],
                lag_ratio=0.08
            ),
            run_time=1.8
        )

        self.wait(0.5)

        # 11. Fill the fraction bits

        fraction_bits = (
            "1001100110011001100110011001100110011001100110011010"
        )

        fraction_source = VGroup(*[
            MathTex(bit, font_size=17, color=BLUE)
            for bit in fraction_bits
        ]).arrange(RIGHT, buff=0.025)

        fraction_source.move_to(
            fraction_slots.get_center() + UP * 0.55
        )

        self.play(
            Write(fraction_source),
            run_time=1.2
        )

        self.play(
            LaggedStart(
                *[
                    bit.animate.move_to(slot.get_center())
                    for bit, slot in zip(fraction_source, fraction_slots)
                ],
                lag_ratio=0.015
            ),
            run_time=3
        )

        self.wait(2)


Overwriting scene.py


In [6]:
!manim -ql scene.py NormalizeBinary

Manim Community v0.22.1

STEP 1: Scene started
[11/20/26 03:49:23] INFO     Animation 0 : Using cached    ]8;id=376310;file:///usr/local/lib/python3.13/dist-packages/manim/renderer/cairo_renderer.py\cairo_renderer.py]8;;\:]8;id=524299;file:///usr/local/lib/python3.13/dist-packages/manim/renderer/cairo_renderer.py#105\105]8;;\
                             data (hash :                                       
                             1653ec4b8b269396fb90d0110a74d                      
                             874be92f172bcfe0513e655ea9322                      
                             785613)                                            
STEP 2: Title completed
                    INFO     Animation 1 : Using cached    ]8;id=701587;file:///usr/local/lib/python3.13/dist-packages/manim/renderer/cairo_renderer.py\cairo_renderer.py]8;;\:]8;id=544068;file:///usr/local/lib/python3.13/dist-packages/manim/renderer/cairo_renderer.py#105\105]8;;\
                           

In [7]:
from IPython.display import Video, display

display(Video(
    "media/videos/scene/480p15/NormalizeBinary.mp4",
    embed=True
))